<center><h1>Velasco_Djovan_HW3</h1></center>
<br>
<br>

Name: Djovan Velasco
<br>
Github Username: djovan-velasco
<br>
USC ID: 8082620160

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os


Get the AReM Data Set

In [5]:
dataset_path = 'AReM'

train_files = []
test_files = []

for activity_folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, activity_folder)
    
    if not os.path.isdir(folder_path):
        continue
        
    files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    
    for file in files:
        file_path = os.path.join(folder_path, file)
        
        if activity_folder in ['bending1', 'bending2']:
            if file in ['dataset1.csv', 'dataset2.csv']:
                test_files.append((activity_folder, file_path))
            else:
                train_files.append((activity_folder, file_path))
                
        else:
            if file in ['dataset1.csv', 'dataset2.csv', 'dataset3.csv']:
                test_files.append((activity_folder, file_path))
            else:
                train_files.append((activity_folder, file_path))


### (b) Test and Train Data

In [ ]:
def load_instance(file_path):
    
    df = pd.read_csv(
        file_path, 
        skiprows=4,
        usecols=range(1, 7)
    )
    
    return df

X_train = []
y_train = []

for activity, file_path in train_files:
    df = load_instance(file_path)
    X_train.append(df)
    y_train.append(activity)

# Rebuild the final testing datasets
X_test = []
y_test = []

for activity, file_path in test_files:
    df = load_instance(file_path)
    X_test.append(df)
    y_test.append(activity)


Loaded 69 training DataFrames.
Native columns preserved: ['avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']


### (c) Feature Extraction

#### i. Research

Common Statistical Time-Domain Features for Time Series Classification: Mean, Median, Variance, Standard Deviation, Skewness, Kurtosis, Minimum, Maximum, Quantiles and Percentiles

#### ii. Extraction

In [11]:
X_all = X_train + X_test 

extracted_features = []
metrics_names = ['min', 'max', 'mean', 'median', 'std', '1st_quart', '3rd_quart']
col_names = []

for i in range(1, 7):
    for m in metrics_names:
        col_names.append(f"{m}{i}")

for df in X_all:
    instance_row = []
    
    for col in df.columns:
        metrics = [
            df[col].min(),
            df[col].max(),
            df[col].mean(),
            df[col].median(),
            df[col].std(),
            df[col].quantile(0.25),
            df[col].quantile(0.75)
        ]
        instance_row.extend(metrics)
        
    extracted_features.append(instance_row)

feature_df = pd.DataFrame(extracted_features, columns=col_names)
print(feature_df.head())
print(feature_df.tail())

    min1   max1      mean1  median1      std1  1st_quart1  3rd_quart1  min2  \
0  36.25  48.00  43.969125    44.50  1.618364       43.31       44.67   0.0   
1  37.00  48.00  43.454958    43.25  1.386098       42.50       45.00   0.0   
2  33.00  47.75  42.179812    43.50  3.670666       39.15       45.00   0.0   
3  33.00  45.75  41.678063    41.75  2.243490       41.33       42.75   0.0   
4  35.00  47.40  43.954500    44.33  1.558835       43.00       45.00   0.0   

   max2     mean2  ...      std5  1st_quart5  3rd_quart5  min6  max6  \
0  1.50  0.413125  ...  3.318301     20.5000       23.75   0.0  2.96   
1  1.58  0.378083  ...  2.488862     22.2500       24.00   0.0  5.26   
2  3.00  0.696042  ...  3.849448     30.4575       36.33   0.0  2.18   
3  2.83  0.535979  ...  2.411026     28.4575       31.25   0.0  1.79   
4  1.70  0.426250  ...  1.999604     35.3625       36.50   0.0  1.79   

      mean6  median6      std6  1st_quart6  3rd_quart6  
0  0.555312     0.49  0.487826     

#### iii. Standard Deviation

In [16]:
n_iterations = 1000
bootstrap_results = []
np.random.seed(123)

for col in feature_df.columns:
    col_data = feature_df[col].astype(float).dropna().values
    n_valid_instances = len(col_data)
    
    actual_std = np.std(col_data, ddof=1)
    
    samples = np.random.choice(col_data, size=(n_iterations, n_valid_instances), replace=True)
    
    sample_stds = np.std(samples, axis=1, ddof=1)
    
    ci_lower = np.percentile(sample_stds, 5)
    ci_upper = np.percentile(sample_stds, 95)
    
    bootstrap_results.append({
        'Feature': col,
        'Estimated_Std': actual_std,
        '90%_CI_Lower': ci_lower,
        '90%_CI_Upper': ci_upper
    })

ci_df = pd.DataFrame(bootstrap_results)
print(ci_df.head(7))
print(ci_df.tail(7))

      Feature  Estimated_Std  90%_CI_Lower  90%_CI_Upper
0        min1       9.624011      8.314955     10.719240
1        max1       4.207745      3.156360      5.154891
2       mean1       5.276431      4.647196      5.830573
3     median1       5.386624      4.733538      5.975009
4        std1       1.771282      1.587891      1.939527
5  1st_quart1       6.127846      5.520238      6.604872
6  3rd_quart1       5.031028      4.189268      5.728203
       Feature  Estimated_Std  90%_CI_Lower  90%_CI_Upper
35        min6       0.046101      0.000000      0.078915
36        max6       2.533515      2.253678      2.762577
37       mean6       1.157222      1.064157      1.215879
38     median6       1.089858      0.991433      1.147937
39        std6       0.516968      0.476362      0.543299
40  1st_quart6       0.761542      0.688332      0.807736
41  3rd_quart6       1.526871      1.402579      1.601901


#### iv. Select Features

Mean, Standard Deviation, Min

## 2. ISLR 3.7.4

### (a) Linear Train

For our training RSS, we would expect our more flexible model to fit best and minimize the RSS. We can think of it as the cubic model fitting the data points much more aggressively or closely than the smoother simple linear model. In this case the cubic model could fit as well or better than the linear model so it would have lower RSS. 

### (b) Linear Test

On the contrary to the training case, testing RSS would be lower for the simple linear model since it captures the true behavior of the data with little bias. The more complex cubic model would overfit the data, have high variance, and thus increase the RSS. 

### (c) Not Linear Train

The cubic model would once again win for the lower training RSS. In this case, while it would fit the data better, in general a flexibile model will map closer to the actual data points than a smooth linear model and have a lower RSS. 

### (d) Not Linear Testing

In this case, we would not have enough information to tell. The trend being simply (non-linear) doesn't give us enough information. If the trend is non-linear but close to linear then our simple linear model would explain the true behavior better and minimize RSS. On the other hand, if the trend is very non-linear (ex: x^3, x^4) then our cubic model would fit the true behavior better and minimize the RSS.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 